# 00. Data Curation & Biological Metadata Enrichment

Robust data curation pipeline for PTM site prediction:
1. **Source Ingestion**: Parses dbPTM / UniProt records for S-glutathionylation, S-nitrosylation, and S-palmitoylation.
2. **Residue Validation**: Strictly enforces that center residue is **Cysteine (`C`)** at position 16 of the 31-mer window.
3. **UniProt Validation**: Coordinates and windows are verified against full canonical sequences fetched via UniProt REST API.
4. **Metadata Extraction**: Enriches each protein with **Taxonomic Lineage** (Taxon ID, organism name) and **Subcellular Localization** for downstream biological splits and feature ablation.

In [ ]:
import os
import time
import requests
import numpy as np
import pandas as pd
from pathlib import Path

print("✓ Libraries imported successfully.")

In [ ]:
# ============================================================================
# CONFIGURATION PARAMETERS
# ============================================================================

ORIGINAL_TRAIN_FILE = "../data/train.csv"
OUTPUT_CLEAN_FILE = "../data/train_curated.csv"
METADATA_STORE_FILE = "../data/protein_metadata_store.csv"

# dbPTM external source files
EXT_FILES = {
    'S-glutathionylation': '../data/Glutathionylation',
    'S-nitrosylation':     '../data/S-nitrosylation',
    'S-palmitoylation':    '../data/S-palmitoylation'
}

WINDOW_SIZE = 31
HALF_WINDOW = 15

print(f"Configuration:")
print(f"  Base Training:  {ORIGINAL_TRAIN_FILE}")
print(f"  Curated Output: {OUTPUT_CLEAN_FILE}")
print(f"  Metadata Store: {METADATA_STORE_FILE}")

In [ ]:
def parse_external_dbptm(ext_files):
    """Parse dbPTM tab-separated positive records."""
    records = []
    for label, path in ext_files.items():
        if not os.path.exists(path):
            print(f"⚠️ File not found: {path}")
            continue
        df = pd.read_csv(path, sep='\t', header=None, dtype=str)
        # Columns: 0=EntryName, 1=UniProtID, 2=Position, 3=PTM, 4=PMID, 5=21mer
        for _, row in df.iterrows():
            pos = pd.to_numeric(row[2], errors='coerce')
            if pd.notna(pos):
                records.append({
                    'EntryName': str(row[0]),
                    'ID': str(row[1]).strip(),
                    'Position': int(pos),
                    'Label': label,
                    'PMID': str(row[4]) if len(row) > 4 else '',
                    'SourceWindow': str(row[5]) if len(row) > 5 else ''
                })
        print(f"  Loaded {len(df):,} records from {label}")
    return pd.DataFrame(records)

df_raw_positives = parse_external_dbptm(EXT_FILES)
print(f"Total raw positive sites parsed: {len(df_raw_positives):,}")

In [ ]:
def fetch_uniprot_batch(accessions, batch_size=300):
    """
    Fetch canonical sequences, taxonomy, and subcellular localization from UniProt REST API.
    """
    base_url = "https://rest.uniprot.org/uniprotkb/accessions"
    metadata_store = {}
    
    unique_ids = sorted(list(set(accessions)))
    print(f"Fetching metadata for {len(unique_ids):,} unique UniProt IDs...")
    
    for i in range(0, len(unique_ids), batch_size):
        batch = unique_ids[i:i + batch_size]
        params = {'accessions': ','.join(batch), 'format': 'json'}
        try:
            resp = requests.get(base_url, params=params, timeout=40)
            if resp.status_code == 200:
                data = resp.json()
                for entry in data.get('results', []):
                    acc = entry.get('primaryAccession')
                    seq = entry.get('sequence', {}).get('value', '')
                    organism = entry.get('organism', {})
                    taxon_id = organism.get('taxonId', None)
                    scientific_name = organism.get('scientificName', '')
                    lineage = organism.get('lineage', [])
                    
                    # Subcellular locations
                    locs = set()
                    for comm in entry.get('comments', []):
                        if comm.get('commentType') == 'SUBCELLULAR LOCATION':
                            for subloc in comm.get('subcellularLocations', []):
                                loc_val = subloc.get('location', {}).get('value')
                                if loc_val:
                                    locs.add(loc_val)
                                    
                    metadata_store[acc] = {
                        'ID': acc,
                        'Sequence': seq,
                        'TaxonID': taxon_id,
                        'Organism': scientific_name,
                        'Kingdom': lineage[0] if lineage else 'Unknown',
                        'SubcellularLocations': '; '.join(sorted(list(locs)))
                    }
        except Exception as e:
            print(f"  Error fetching batch {i}: {e}")
        time.sleep(0.5)
        
    print(f"✓ Retrieved metadata for {len(metadata_store):,} proteins.")
    return metadata_store

In [ ]:
def extract_validated_31mers(df_sites, protein_dict):
    """
    Extract 31-residue windows centered on verified Cysteine residues.
    Quarantines non-cysteine centers and out-of-bound coordinates.
    """
    validated_rows = []
    quarantined = []
    
    for _, row in df_sites.iterrows():
        uid = row['ID']
        pos = row['Position'] # 1-based
        label = row['Label']
        
        if uid not in protein_dict:
            quarantined.append({**row.to_dict(), 'Reason': 'Protein sequence not found'})
            continue
            
        full_seq = protein_dict[uid]['Sequence']
        center_0idx = pos - 1
        
        if center_0idx < 0 or center_0idx >= len(full_seq):
            quarantined.append({**row.to_dict(), 'Reason': 'Coordinate out of bounds'})
            continue
            
        # STRICT RESIDUE VALIDATION: Must be Cysteine
        target_aa = full_seq[center_0idx]
        if target_aa != 'C':
            quarantined.append({**row.to_dict(), 'Reason': f'Non-cysteine target: {target_aa}'})
            continue
            
        # Extract 31-mer window (+/- 15 residues)
        start = center_0idx - 15
        end = center_0idx + 16
        
        left_pad = "X" * max(0, -start)
        right_pad = "X" * max(0, end - len(full_seq))
        
        valid_start = max(0, start)
        valid_end = min(len(full_seq), end)
        
        window = left_pad + full_seq[valid_start:valid_end] + right_pad
        
        if len(window) == 31 and window[15] == 'C':
            validated_rows.append({
                'ID': uid,
                'Sequence': window,
                'Position': pos,
                'Label': label
            })
            
    print(f"Validation summary:\n  Accepted positive sites: {len(validated_rows):,}\n  Quarantined sites:       {len(quarantined):,}")
    return pd.DataFrame(validated_rows), pd.DataFrame(quarantined)

In [ ]:
def merge_and_export_curated_data(original_train_path, df_valid_sites, meta_store, output_path, meta_output_path):
    """Merge validated external sites with base train.csv, deduplicate, and attach metadata."""
    print(f"Merging with {original_train_path}...")
    df_base = pd.read_csv(original_train_path)
    
    label_cols = ['S-glutathionylation', 'S-nitrosylation', 'S-palmitoylation']
    
    # Format valid external sites into multi-label pivot
    if not df_valid_sites.empty:
        df_pivoted = df_valid_sites.pivot_table(
            index=['ID', 'Sequence'],
            columns='Label',
            aggfunc='size',
            fill_value=0
        ).reset_index()
        
        for col in label_cols:
            if col not in df_pivoted.columns:
                df_pivoted[col] = 0
            else:
                df_pivoted[col] = (df_pivoted[col] > 0).astype(int)
                
        df_combined = pd.concat([df_base[label_cols + ['ID', 'Sequence']], df_pivoted[label_cols + ['ID', 'Sequence']]], ignore_index=True)
    else:
        df_combined = df_base
        
    # Group by ID and Sequence, keeping maximum label indicator (positives override negatives)
    df_clean = df_combined.groupby(['ID', 'Sequence'], as_index=False)[label_cols].max()
    
    # Strictly filter any non-cysteine centers from the final output
    df_clean = df_clean[df_clean['Sequence'].str[15] == 'C'].reset_index(drop=True)
    
    df_clean.to_csv(output_path, index=False)
    print(f"✓ Saved clean curated dataset to: {output_path} ({len(df_clean):,} samples)")
    
    # Save metadata table
    if meta_store:
        df_meta = pd.DataFrame(list(meta_store.values()))
        df_meta.to_csv(meta_output_path, index=False)
        print(f"✓ Saved protein metadata store to: {meta_output_path} ({len(df_meta):,} entries)")
        
    return df_clean